## Analysis Notebook for AI Sycophancy and Prompt Similarity

Given $N$ prompt items, for each item $i \in \set{1, ..., N}$:
- The Reddit AITA prompt = $p_i$
- The LLM's response to that prompt = $r_i^{LLM}$
- LLM syccophancy label = $y_i^{LLM}\in\set{0, 1}$
- The most upvoted crowd response to that prompt = $r_i^C$
Since divergence from upvoted responses is used to flag sycophancy (elephant metric), we assume the most upvoted response as a ground truth label


Asssumptions:
- $r_i^C$ is treated as a ground truth human response for prompt $i$
- Divergence from $r_i^C$ is a syccophancy flag


**Research Questions:**

1. Are prompts whose LLM responses are labeled sycophantic more semantically similar to one another than prompts whose LLM responses are labeled non-sycophantic?
    1. Are there perceptible clusters in sycophancy-generated prompts, respective to each model?
2. Are LLM responses labeled sycophantic less semantically similar to the most upvoted crowd response than non-sycophantic LLM responses?


**Metric Definitions (Can Move these to appropriate sections in notebook)**:

1. *Prompt Similarity*

We use SBERT to embed prompts

$$e_i^p = SBERT(p_i)$$

And define pairwise cosine similarity

$$s_{ij}^p = cos(e_i^p, e_j^p)$$

2. *LLM to Crowd Source Sycophancy*

We embed LLM and Crowd-sourced responses using SBERT

$$e_i^{LLM} = SBERT(r_i^{LLM}), e_i^C = SBERT(r_i^C)$$

Then define response similarity as $cos(e_i^{LLM},e_i^C)$


3. Robustness Check with Bertscore (Optional, omit if takes too long to run all pairs) F1. 

### Imports

In [1]:
import logging
from pathlib import Path

import pandas as pd
import transformers

from analysis.bert_score import BERTScoreScorer
from analysis.data import load_llm_crowd_pairs
from db.crud import ensure_system_prompt
from db.database import get_session
from models import Model
from prompts import SystemPrompt

transformers.tokenization_utils.logger.setLevel(logging.ERROR)
transformers.configuration_utils.logger.setLevel(logging.ERROR)
transformers.modeling_utils.logger.setLevel(logging.ERROR)

c:\Users\mruss\projects\classes\ds587\ai-sycophancy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASETS_DIR = Path('datasets')

## LLM-to-Crowd Response Divergence

Partition the prompt items by the model-specific sycophancy label for each model.

For each model, $m$ define the sycophantic (+) and nonsycophantic subsets (-):

$$P_{m,+}=\set{i:y_{im}=1}, \, P_{m,-}=\set{i:y_{im}=0}$$

For each prompt item $i$ and model $m$:

- the model response is $r_{im}$
- the most upvoted crowd response is $r_i^C$

We treat the most upvoted crowd response $r_i^C$ as the reference human response for prompt $i$.

> *NOTE*: This should all largely overlap with Part 1 work. Will be stored in a CSV

### SBERT Crowd Divergence

Define the SBERT-based crowd divergence for each prompt-model pair:

$$d_{im}^{SBERT}=1-\cos(\text{SBERT}(r_{im}), \text{SBERT}(r_i^C))$$

Here:

- $d_{im}^{SBERT}\approx 0$: the model response is very close to the crowd-endorsed response
- larger values of $d_{im}^{SBERT}$ mean the model response deviates more from the crowd-endorsed response

Define the corresponding mean crowd divergence within each group:

$$
D_{m,+}^{SBERT}=\frac{1}{|P_{m,+}|}\sum_{i\in P_{m,+}} d_{im}^{SBERT}\\
D_{m,-}^{SBERT}=\frac{1}{|P_{m,-}|}\sum_{i\in P_{m,-}} d_{im}^{SBERT}
$$

Then define the model-specific excess crowd divergence:

$$\Gamma_{\text{crowd}}^{SBERT}(m)=D_{m,+}^{SBERT}-D_{m,-}^{SBERT}$$

**Interpretation**:

- $>0$: responses labeled sycophantic diverge more from the crowd-endorsed response than responses labeled non-sycophantic
- $\approx 0$: little or no difference in crowd divergence between the two groups
- $<0$: responses labeled sycophantic diverge less from the crowd-endorsed response than responses labeled non-sycophantic

### BERTScore Robustness Check

As a secondary robustness check, define the BERTScore-based crowd divergence for each prompt-model pair:

$$d_{im}^{BERT}=1-\text{BERTScore-F1}(r_{im}, r_i^C)$$

Define the corresponding mean BERTScore crowd divergence within each group:

$$
D_{m,+}^{BERT}=\frac{1}{|P_{m,+}|}\sum_{i\in P_{m,+}} d_{im}^{BERT}\\
D_{m,-}^{BERT}=\frac{1}{|P_{m,-}|}\sum_{i\in P_{m,-}} d_{im}^{BERT}
$$

Then define the model-specific excess crowd divergence under BERTScore:

$$\Gamma_{\text{crowd}}^{BERT}(m)=D_{m,+}^{BERT}-D_{m,-}^{BERT}$$

**Interpretation**:

- $>0$: responses labeled sycophantic diverge more from the crowd-endorsed response under BERTScore
- $\approx 0$: little or no difference in divergence under BERTScore
- $<0$: responses labeled sycophantic diverge less from the crowd-endorsed response under BERTScore

### Recommended reporting

For each model $m$, report:

- $D_{m,+}^{SBERT}$
- $D_{m,-}^{SBERT}$
- $\Gamma_{\text{crowd}}^{SBERT}(m)$

and optionally:

- $D_{m,+}^{BERT}$
- $D_{m,-}^{BERT}$
- $\Gamma_{\text{crowd}}^{BERT}(m)$

### Optional exploratory analyses

To assess whether sycophancy is associated with large crowd divergence patterns across models, optionally compute:

- boxplots or violin plots of $d_{im}^{SBERT}$ by $y_{im}$ for each model
- bootstrap confidence intervals for each $\Gamma_{\text{crowd}}^{SBERT}(m)$
- the same visual summaries for BERTScore as a robustness check

These are exploratory and should be presented as supporting evidence rather than core inferential results.